# Where does a model go wrong? Error analysis on UCI Adult

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toclocoinc/lattice-grid-notebooks/blob/main/model-error-adult/model_error_adult.ipynb)

Train a gradient-boosted classifier on the census "Adult" data, then look at its mistakes in a grid instead of a metrics table: every test row with its label, score and features, a **colour scale on the error**, an **actual x predicted pivot**, **grouping by sex, race or age band** to see where the error concentrates, and a **ROC chart bound to the grid** that redraws for whatever segment you filter or select. The rows you pick come back as a pandas `DataFrame` for the next iteration. No API key.

**Data:** [UCI Adult (Census Income)](https://archive.ics.uci.edu/dataset/2/adult), Becker & Kohavi, 1996, licensed **CC BY 4.0**; fetched from the UCI archive when you run the notebook (about 6 MB). The notebook code is MIT-licensed (see the repo `LICENSE`).

In [1]:
%pip install -q "lattice-grid-jupyter>=1.111.3.0"

## 1. Fetch the data and fit a model

The model never sees `sex` or `race`, so any gap between groups is something it picked up from the other columns. We fit on the official training file and score the official test file (16,281 people).

In [2]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

UCI = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/"
names = ["age", "workclass", "fnlwgt", "education", "education_num", "marital_status", "occupation", "relationship",
         "race", "sex", "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]
read = lambda f, **kw: pd.read_csv(UCI + f, names=names, skipinitialspace=True, na_values="?", **kw)
train, test = read("adult.data"), read("adult.test", skiprows=1)
cats = ["workclass", "marital_status", "occupation", "relationship", "native_country"]
features = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"] + cats
for c in cats:  # same category codes in train and test
    levels = sorted(train[c].dropna().unique())
    train[c], test[c] = pd.Categorical(train[c], levels), pd.Categorical(test[c], levels)

model = HistGradientBoostingClassifier(categorical_features="from_dtype", random_state=0)
model.fit(train[features], train.income.str.startswith(">50K"))
score = model.predict_proba(test[features])[:, 1]
print(f"{len(test):,} test rows, accuracy {((score >= .5) == test.income.str.startswith('>50K')).mean():.3f}")

16,281 test rows, accuracy 0.872


## 2. The predictions frame

One row per person: the true `label` (1 = earns over $50K), the model's `score`, what it `predicted` at a 0.5 cut-off, the `error` (how far the score is from the truth, 0 to 1), and the features to slice by (`sex`, `race`, `age_band`).

In [3]:
preds = test[["age", "sex", "race"]].copy()
preds["age_band"] = pd.cut(test.age, [0, 25, 35, 45, 55, 65, 120], labels=["<=25", "26-35", "36-45", "46-55", "56-65", "66+"]).astype(str)
preds["label"] = test.income.str.startswith(">50K").astype(int)
preds["score"] = score.round(3)
preds["error"] = (preds.label - preds.score).abs().round(3)
preds["actual"] = preds.label.map({0: "<=50K", 1: ">50K"})
preds["predicted"] = (preds.score >= .5).map({False: "<=50K", True: ">50K"})
preds["wrong"] = preds.actual != preds.predicted
preds = preds[["sex", "age_band", "race", "actual", "predicted", "label", "score", "error", "wrong"]]
preds = preds.reset_index(drop=True)
preds.head(3)

,sex,age_band,race,actual,predicted,label,score,error,wrong
0,Male,<=25,Black,<=50K,<=50K,0,0.003,0.003,False
1,Male,36-45,White,<=50K,<=50K,0,0.156,0.156,False
2,Male,26-35,White,>50K,<=50K,1,0.286,0.714,True


## 3. Grid and ROC chart

The `error` column carries a **red colour scale** (white = the model was sure and right, red = sure and wrong). The grid opens **grouped by sex** with the average error on each group row. Open the left panel's **Columns** tab to drag `race` or `age_band` into the row groups instead.

The chart is **bound to the grid**: pick a segment in the dropdown (or use the grid's own column filters or the left **Filters** tab) and the ROC curve and its AUC redraw in the browser for just those rows, with no round trip to Python. Compare `age_band = 66+` with `age_band = 36-45`, or `sex = Female` with `sex = Male`.

In [4]:
from ipywidgets import Dropdown, HBox, Layout, VBox
from lattice_grid_jupyter import LatticeGridWidget, LatticeChart

WIDTHS = dict(sex=150, age_band=95, race=75, actual=60, predicted=80, score=60, error=150, wrong=70)
scale = {"scale": {"min": 0, "max": 1, "colours": ["#ffffff", "#d64545"]}}
grid = LatticeGridWidget(preds, height=520, options={"formatting": {"error": [scale]}, "toolPanel": {"panels": ["columns", "filters"]}},
    columns=[{"field": f, "layout": {"width": w}} for f, w in WIDTHS.items()] + [{"field": "label", "layout": {"hidden": True}}, {"field": "error", "total": "avg", "format": {"decimals": 3}}],
    state={"group": ["sex"]})
roc = LatticeChart(grid, type="roc", label="label", score="score", height=520)
grid.layout, roc.layout = Layout(width="70%"), Layout(width="30%")

segments = ["All people"] + [f"{c} = {v}" for c in ("sex", "race", "age_band") for v in sorted(preds[c].unique())]
pick = Dropdown(options=segments, description="Segment")
def show(change):  # a segment is just a grid filter: the bound chart redraws for it
    col, _, val = change.new.partition(" = ")
    grid.state = {**grid.state, "filters": {"op": "and", "conditions": [{"col": col, "op": "eq", "value": val}] if val else []}}
pick.observe(show, "value")
VBox([pick, HBox([grid, roc])])

![the grid and ROC chart](https://raw.githubusercontent.com/toclocoinc/lattice-grid-notebooks/main/model-error-adult/grid.png)

*The grid in JupyterLab, grouped by sex: each group row reads its label and count (`Female (5421)`) with the average error, the error column is on its red scale, and the ROC chart (AUC 0.927 for everyone) sits beside it. Pick `age_band = 66+` and it redraws for just those people (AUC 0.857).*

## 4. Actual x predicted

A second grid over the same frame, pivoted: actual class down the side, predicted class across the top, and the number of people in each cell. The off-diagonal cells are the model's mistakes.

In [5]:
cells = LatticeGridWidget(preds[["actual", "predicted", "sex", "error"]], height=150, pivot=["predicted"],
    columns=[{"field": "error", "total": "count", "header": "people"}], state={"group": ["actual"]})
cells

## 5. Pick the wrong rows, get a DataFrame

In the first grid, filter `wrong` to `true` (or select any rows you like) and run the next cell: `grid.selected` is the selection as a pandas `DataFrame` with its original index and dtypes. If nothing is selected it falls back to the grid's filtered view, and to every wrong row when the view is unfiltered, so the notebook also runs top to bottom unattended. From here it is ordinary pandas: the table below splits the mistakes into people the model missed (actual `>50K`) and people it over-called.

In [6]:
rows = grid.selected
if rows.empty:
    rows = grid.view
rows = rows[rows.wrong]
print(f"{len(rows):,} wrong rows -> DataFrame")
rows.groupby(["actual", "sex"]).agg(wrong=("error", "size"), avg_error=("error", "mean"), avg_score=("score", "mean")).round(2)

2,084 wrong rows -> DataFrame


wrong  avg_error  avg_score
actual sex                                
<=50K  Female    115       0.64       0.64
       Male      613       0.65       0.65
>50K   Female    233       0.77       0.23
       Male     1123       0.71       0.29